## Backup runner (headless)

Runs one lakehouse backup with **no widgets and no typed names**, so it can be started by the Fabric **Run On Demand Notebook Job** API (for example from the Rayfin admin app). It does the same thing as `nb_backup_form` + `nb_backup_executor`, on the same engine (`nb_backup_restore_adls_v6`, loaded with `%run`, so it **must be in this same workspace**).

- **Parameters** (first code cell, tagged *parameters*): IDs, archive address, folder, two switches, a `request_key`.
- **Checks before anything is written** (same as the form, now enforced): all IDs are GUIDs, staging is not the source, both lakehouses exist and are readable, the container can be listed (**blocking**, the form only warned).
- **Result**: one JSON document, returned with `notebookutils.notebook.exit(...)`, printed as `RESULT_JSON ...`, optionally also written to a control lakehouse. A failed run ends the job as *Failed* unless `fail_on_error` is false.
- Running it interactively with the defaults stops at the first check (empty IDs) and changes nothing.

In [ ]:
# Parameters. This cell is tagged "parameters": the values are defaults, and the Run On Demand job API
# (or a pipeline / notebookutils.notebook.run) overrides them by name. Names are case-sensitive in the code.
# Send IDs, not names. Types in the job request: Text -> str, Boolean -> bool, Integer -> int.
source_workspace_id = ""          # Text   GUID of the workspace that holds the lakehouse to back up
source_lakehouse_id = ""          # Text   GUID of the lakehouse to back up
staging_workspace_id = ""         # Text   GUID of the workspace that holds the staging lakehouse
staging_lakehouse_id = ""         # Text   GUID of the staging lakehouse (must not be the source)
archive_root_uri = ""             # Text   abfss://<container>@<account>.dfs.core.windows.net
backup_folder = ""                # Text   folder inside the container ("" = the container root)
copy_files = True                 # Boolean  include everything under Files (form default: on)
verify_counts = True              # Boolean  also compare row counts (form default: on)
only_schemas = ""                 # Text   optional, comma separated; blank = all schemas
only_tables = ""                  # Text   optional, comma separated; blank = all tables
backup_timestamp = ""             # Text   optional 12 digits YYYYMMDDHHMM; blank = now (UTC). Set it to make a retry idempotent
request_key = ""                  # Text   caller's id for tracing / idempotency; blank = a manual-xxxx id is made
result_workspace_id = ""          # Text   optional: workspace of a control lakehouse to receive the result JSON file
result_lakehouse_id = ""          # Text   optional: control lakehouse (Files/rayfin_runs/...); must not be the source
fail_on_error = True              # Boolean  True: a failed run ends the job as Failed (JSON in the error text); False: always exit with JSON

### Load the engine (same workspace)

In [ ]:
%run nb_backup_restore_adls_v6

### Helpers

In [ ]:
# ---- shared runner helpers (identical in nb_backup_run and nb_restore_run; engine is not touched) ----
import contextlib
import json
import logging
import re
import sys
import uuid
from datetime import datetime, timezone

RUNNER = "nb_backup_run"
SCHEMA_VERSION = 1
LIST_CAP = 200                       # long lists in the exit JSON are cut here (the manifest in the archive has all)
_STARTED = datetime.now(timezone.utc)
_GUID = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")


class RunnerError(Exception):
    """A problem the caller can act on. `code` is stable and machine-readable."""
    def __init__(self, code, message):
        super().__init__(message)
        self.code = code
        self.message = message


def _guid(value, label):
    text = str(value).strip().lower()
    if not _GUID.fullmatch(text):
        raise RunnerError("invalid_parameter", f"{label} must be a GUID, got {str(value)!r}")
    return text


def _bool(value, label):
    """Accepts real booleans; also the strings 'true'/'false' in case a caller sent Text instead of Boolean."""
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in ("true", "false"):
        return value.strip().lower() == "true"
    raise RunnerError("invalid_parameter", f"{label} must be true or false, got {value!r}")


def _request_key(value):
    text = str(value).strip()
    if not text:
        text = "manual-" + uuid.uuid4().hex[:8]
    if not re.fullmatch(r"[A-Za-z0-9._:-]{1,128}", text):
        raise RunnerError("invalid_parameter", "request_key may only contain letters, digits, . _ : - (max 128)")
    return text


def _archive_uri(archive_root_uri, backup_folder):
    """Same address rule as the forms (abfss form only) plus the optional single folder."""
    uri = str(archive_root_uri).strip().rstrip("/")
    if not re.fullmatch(r"abfss://[^@/\s]+@[^./\s]+\.dfs\.core\.windows\.net(/\S*)?", uri):
        raise RunnerError("invalid_parameter",
                          "archive_root_uri must look like abfss://<container>@<account>.dfs.core.windows.net")
    folder = str(backup_folder).strip().strip("/")
    if folder:
        for part in folder.split("/"):
            if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", part):
                raise RunnerError("invalid_parameter",
                                  "backup_folder may only contain letters, digits, dot, dash and underscore")
        uri = f"{uri}/{folder}"
    if any(part in (".", "..") for part in uri.split("//", 1)[1].split("/")):
        raise RunnerError("invalid_parameter", "archive_root_uri must not contain . or .. segments")
    return uri


def _name_set(text, label):
    """'dbo, sales' -> {'dbo', 'sales'}. Blank -> None, which means everything."""
    names = {part.strip() for part in str(text).split(",") if part.strip()}
    return names or None


def _container_reachable(archive):
    """Blocking version of the form's check: the container itself must be listable."""
    root = re.match(r"abfss://[^/]+", archive).group(0)
    try:
        fs.ls(root)
    except Exception as exc:
        raise RunnerError("container_unreachable",
                          f"Cannot read the container {root}. Check the address and the runner's access. "
                          + " ".join(str(exc).split())[:200])


def _lakehouse(workspace_id, lakehouse_id, label):
    """Engine lookup (Fabric REST, runner identity). Proves the IDs exist and are readable."""
    try:
        return lakehouse_info(workspace_id, lakehouse_id)
    except Exception as exc:
        raise RunnerError("lakehouse_not_found",
                          f"{label}: workspace {workspace_id} / lakehouse {lakehouse_id} was not found, "
                          f"or the runner has no access to it. {' '.join(str(exc).split())[:200]}")


def _cap(items):
    items = list(items)
    return items[:LIST_CAP], len(items)


class _Tee:
    """Copies what the engine prints into a list while still passing it through."""
    def __init__(self, stream):
        self._stream, self.lines, self._buf = stream, [], ""

    def write(self, text):
        self._buf += text
        while "\n" in self._buf:
            line, self._buf = self._buf.split("\n", 1)
            self.lines.append(line)
        return self._stream.write(text)

    def flush(self):
        return self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


class _LogCollector(logging.Handler):
    """Collects the engine's log messages (its logger is `logger`) so a failure can report where it stopped."""
    def __init__(self):
        super().__init__(logging.INFO)
        self.messages = []

    def emit(self, record):
        self.messages.append(f"{record.levelname} {record.getMessage()}")


@contextlib.contextmanager
def _capture():
    """Capture engine prints and log lines for the duration of one engine call."""
    tee, collector = _Tee(sys.stdout), _LogCollector()
    logger.addHandler(collector)
    try:
        with contextlib.redirect_stdout(tee):
            yield tee, collector
    finally:
        logger.removeHandler(collector)


def _base_result():
    return {"schema_version": SCHEMA_VERSION, "runner": RUNNER, "request_key": None,
            "status": "failed", "error": None,
            "started_utc": _STARTED.isoformat(timespec="seconds")}


def _failure(result, code, message, log=None, report=None):
    result["status"] = "failed"
    result["error"] = {"code": code, "message": str(message)[:1500]}
    if log:
        result["log_tail"] = list(log)[-25:]
    if report:
        result["report"] = list(report)[-25:]
    return result


def _finish(result):
    finished = datetime.now(timezone.utc)
    result["finished_utc"] = finished.isoformat(timespec="seconds")
    result["duration_seconds"] = round((finished - _STARTED).total_seconds(), 1)
    return result


def _write_result_file(result, result_workspace_id, result_lakehouse_id):
    """Optional: the job API does not return the exit value, so the runner can also drop the JSON in a
    control lakehouse the app can read: Files/rayfin_runs/<runner>/<request_key>.json. Best effort."""
    if not (str(result_workspace_id).strip() and str(result_lakehouse_id).strip()):
        return None
    try:
        path = (f"{lakehouse_root(_guid(result_workspace_id, 'result_workspace_id'), _guid(result_lakehouse_id, 'result_lakehouse_id'))}"
                f"/Files/rayfin_runs/{RUNNER}/{result['request_key']}.json")
        fs.put(path, json.dumps(result, default=str, indent=2), True)
        return path
    except Exception as exc:
        logger.warning("Could not write the result file: %s", reason(exc))
        return None

### Validation (the form's guards, server side)

In [ ]:
def validate_backup_params():
    """Server-side version of the Backup form's guards. Reads only. Returns the clean values to run with."""
    v = {}
    v["src_ws"] = _guid(source_workspace_id, "source_workspace_id")
    v["src_lh"] = _guid(source_lakehouse_id, "source_lakehouse_id")
    v["stg_ws"] = _guid(staging_workspace_id, "staging_workspace_id")
    v["stg_lh"] = _guid(staging_lakehouse_id, "staging_lakehouse_id")
    if (v["src_ws"], v["src_lh"]) == (v["stg_ws"], v["stg_lh"]):
        raise RunnerError("staging_is_source", "The staging lakehouse must not be the source lakehouse.")
    v["copy_files"] = _bool(copy_files, "copy_files")
    v["verify_counts"] = _bool(verify_counts, "verify_counts")
    v["fail_on_error"] = _bool(fail_on_error, "fail_on_error")
    v["archive"] = _archive_uri(archive_root_uri, backup_folder)
    ts = str(backup_timestamp).strip()
    if ts and not re.fullmatch(r"\d{12}", ts):
        raise RunnerError("invalid_parameter", f"backup_timestamp must be 12 digits (YYYYMMDDHHMM), got {ts!r}")
    v["ts"] = ts or None
    v["schemas"] = _name_set(only_schemas, "only_schemas")
    v["tables"] = _name_set(only_tables, "only_tables")
    if str(result_workspace_id).strip() or str(result_lakehouse_id).strip():
        rw, rl = _guid(result_workspace_id, "result_workspace_id"), _guid(result_lakehouse_id, "result_lakehouse_id")
        if (rw, rl) == (v["src_ws"], v["src_lh"]):
            raise RunnerError("invalid_parameter", "The result lakehouse must not be the source lakehouse.")
    # lookups: the IDs must exist and be readable by the runner identity
    v["src_info"] = _lakehouse(v["src_ws"], v["src_lh"], "Source lakehouse")
    v["stg_info"] = _lakehouse(v["stg_ws"], v["stg_lh"], "Staging lakehouse")
    _container_reachable(v["archive"])                       # the form only warned; here it blocks
    return v

### Run

In [ ]:
RESULT = _base_result()
_report, _log = [], []
try:
    RESULT["request_key"] = _request_key(request_key)
    V = validate_backup_params()
    RESULT["source"] = {"workspace_id": V["src_ws"], "lakehouse_id": V["src_lh"],
                        "workspace_name": V["src_info"]["workspace_name"],
                        "lakehouse_name": V["src_info"]["lakehouse_name"]}
    RESULT["archive"] = V["archive"]

    # Idempotent retry: the caller fixed the timestamp and that backup already completed for this source.
    _existing = None
    if V["ts"]:
        _manifest_path = manifest_path_of(V["archive"], V["ts"])
        if fs.exists(_manifest_path):
            _existing = read_json(_manifest_path)
            if _existing.get("status") != "complete" or _existing.get("source", {}).get("lakehouse_id") != V["src_lh"]:
                raise RunnerError("timestamp_conflict",
                                  f"A backup {V['ts']} already exists in {V['archive']} and is not a complete "
                                  "backup of this source lakehouse. Use a new backup_timestamp.")

    if _existing is not None:
        _ts, _manifest = V["ts"], _existing
        RESULT["already_existed"] = True
    else:
        with _capture() as (_tee, _logs):
            try:
                _ts = backup_lakehouse_to_archive(
                    V["src_ws"], V["src_lh"], V["archive"],
                    staging_ws_id=V["stg_ws"], staging_lh_id=V["stg_lh"],
                    backup_timestamp=V["ts"],
                    copy_files=V["copy_files"], verify_counts=V["verify_counts"],
                    schemas=V["schemas"], tables=V["tables"],
                )
            finally:
                _report, _log = _tee.lines, _logs.messages
        _manifest = None
        RESULT["already_existed"] = False
        try:
            _manifest = read_json(manifest_path_of(V["archive"], _ts))
        except Exception as exc:                              # the engine already verified it; summary only
            logger.warning("Could not re-read the manifest for the summary: %s", reason(exc))

    RESULT["status"] = "succeeded"
    RESULT["backup_timestamp"] = _ts
    RESULT["backup_location"] = run_root_of(V["archive"], _ts)
    if _manifest is not None:
        _tables = _manifest.get("tables", [])
        RESULT["tables_backed_up"] = len(_tables)
        RESULT["schemas"] = sorted({t["schema"] for t in _tables})
        RESULT["rows_total"] = sum(t.get("rows", 0) for t in _tables) if _manifest.get("verify_counts") else None
        RESULT["files_included"] = bool(_manifest.get("files_copied"))
        RESULT["verify_counts"] = bool(_manifest.get("verify_counts"))
        RESULT["copy_method"] = _manifest.get("copy_method")
        RESULT["skipped_shortcuts"], RESULT["skipped_shortcuts_count"] = _cap(_manifest.get("skipped_shortcuts", []))
        RESULT["skipped_non_delta"], RESULT["skipped_non_delta_count"] = _cap(_manifest.get("skipped_non_delta", []))
        RESULT["manifest_completed_utc"] = _manifest.get("completed_utc")
    if _report:
        RESULT["report"] = _report[-10:]
except RunnerError as exc:
    _failure(RESULT, exc.code, exc.message, _log, _report)
except Exception as exc:
    _failure(RESULT, "backup_failed", reason(exc), _log, _report)
_finish(RESULT)

### Return the result

In [ ]:
# Result delivery. Do not wrap notebookutils.notebook.exit in try/except: the exit would not take effect.
_result_file = (_write_result_file(RESULT, result_workspace_id, result_lakehouse_id)
                if RESULT.get("request_key") else None)
if _result_file:
    RESULT["result_file"] = _result_file
_payload = json.dumps(RESULT, default=str, separators=(",", ":"))
print("RESULT_JSON " + _payload)
if RESULT["status"] == "failed" and str(fail_on_error).strip().lower() != "false":
    raise RuntimeError("RAYFIN_RESULT " + _payload)          # job ends as Failed; failureReason carries the JSON
notebookutils.notebook.exit(_payload)